In [1]:
import pandas as pd
df = pd.read_csv("train.csv")

#پرکردن خانه های خالی از جنس استرینگ
df["PoolQC"] = df["PoolQC"].fillna("No Pool")
df["MiscFeature"] = df["MiscFeature"].fillna("None")
df["Alley"] = df["Alley"].fillna("No alley access")
df["Fence"] = df["Fence"].fillna("No Fence")
df["MasVnrType"] = df["MasVnrType"].fillna("None")
df["FireplaceQu"] = df["FireplaceQu"].fillna("No Fireplace")
df["GarageQual"] = df["GarageQual"].fillna("No Garage")
df["GarageFinish"] = df["GarageFinish"].fillna("No Garage")
df["GarageType"] = df["GarageType"].fillna("No Garage")
df["GarageCond"] = df["GarageCond"].fillna("No Garage")
df["BsmtFinType2"] = df["BsmtFinType2"].fillna("No Basement")
df["BsmtExposure"] = df["BsmtExposure"].fillna("No Basement")
df["BsmtCond"] = df["BsmtCond"].fillna("No Basement")
df["BsmtQual"] = df["BsmtQual"].fillna("No Basement")
df["BsmtFinType1"] = df["BsmtFinType1"].fillna("No Basement")
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

#پرکردن خانه های خالی از جنس عدد
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())

df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)

df["MasVnrArea"] = df["MasVnrArea"].fillna(0)

#تعیین x,y
X = df.drop(["Id", "SalePrice"], axis=1)
y = df["SalePrice"]

#تبدیل داده استرینگ به عدد
X = pd.get_dummies(X)

#تقسیم دیتا به دو بخش آموزش و تست
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#انتخاب مدل
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

#آموزش 
model.fit(X_train, y_train)

####################################################
#هدف اینه که ببینیم آیا مدل میتونی پیش بینی خوبی انجام بده و کمک کنیم بتونه پیش بینی های بهتری انجام بده
#حالا میخوام روی دیتای تست همونی که اول جدا کرده بودم 20 درصد دیتاها ، پیش بینی انجام بده
predictions = model.predict(X_test)

#بعد مقایسه میکنی با مقادیر واقعی برای خروجی دیتای تست
comparison = pd.DataFrame({
    "Actual": y_test,
    "Predicted": predictions
})

comparison.head(10)

,Actual,Predicted
892,154500,141143.00
1105,325000,332068.45
413,115000,116192.25
522,159000,153087.00
1036,315500,325493.39
614,75500,83811.00
218,311500,209452.65
1160,146000,149981.30
649,84500,82113.66
887,135500,131265.21


In [2]:
#اولین سوال اینه آیا مدل اصلاً چیزی یاد گرفته؟
#اگر مدل هیچ چیز یاد نگرفته بود، ممکن بود برای همه خانه‌ها یک عدد ثابت بگوید

#سؤال دوم: آیا پیش‌بینی‌ها نزدیک هستند؟
#نمونه اول حدود ۱۳ هزار دلار اختلاف.برای خانه‌ای با قیمت حدود ۱۵۰ هزار دلار، این اختلاف خیلی زیاد نیست.
#نمونه دوم تقریباً عالی.فقط حدود ۵ هزار دلار اختلاف دارد.
#نمونه سوم کمتر از هزار دلار اختلاف.این هم پیش‌بینی بسیار خوبی است.


#سؤال سوم: آیا مدل همیشه خوب پیش‌بینی کرده؟
#نه به این ردیف (218) نگاه کن
# واقعی:311500
#پیش‌بینی:   212160
#اینجا تقریباً ۱۰۰ هزار دلار اختلاف وجود دارد.
#این یعنی برای این خانه، مدل نتوانسته الگوی درستی پیدا کند.

#سؤال چهارم: چرا فقط یک خانه این‌قدر خطا دارد؟
# اگر اختلاف پیش‌بینی و مقدار واقعی زیاد باشد، ممکن است آن خانه ویژگی‌های خاصی
# داشته باشد که نمونه‌های مشابه آن در داده‌های آموزشی کم بوده‌اند؛ بنابراین مدل
# نتوانسته الگوی آن را به‌خوبی یاد بگیرد و پیش‌بینی دقت کمتری داشته است.

In [3]:
X_test.loc[218]

MSSubClass                  50
LotFrontage               69.0
LotArea                  15660
OverallQual                  7
OverallCond                  9
                         ...  
SaleCondition_AdjLand    False
SaleCondition_Alloca     False
SaleCondition_Family     False
SaleCondition_Normal      True
SaleCondition_Partial    False
Name: 218, Length: 302, dtype: object

In [4]:
#چیزی که تو نتایج بالا برای مورد 218 اومده اینه که 
#OverallQual = 7 (خوبه، ولی فوق‌العاده نیست)
#GrLivArea = 1954 (متراژ نسبتاً بزرگ)
#OverallCond = 9 (وضعیت نگهداری عالی)
#Neighborhood_Crawfor = True ⭐⭐⭐چیزی که از همه بیشتر جلب توجه می‌کند:
#YearRemodAdd = 2006 (بازسازی شده)

#محله Crawfor در این دیتاست یکی از محله‌های گران‌تر است.
#اگر Random Forest نتوانسته اثر این محله را خوب یاد بگیرد، ممکن است قیمت را خیلی پایین حدس بزند.

In [5]:
#واسه اینکه ببینیم چرا این مدل رو این خونه همچین اشتباهی کرده یه کاری میکنم
#می‌خواهم به جای اینکه فقط یک خانه را ببینیم، ببینیم مدل روی چه نوع خانه‌هایی بیشتر اشتباه می‌کند.
comparison["Error"] = comparison["Actual"] - comparison["Predicted"]
comparison["AbsError"] = comparison["Error"].abs() #اگر مدل 100 هزار دلار بیشتر یا کمتر پیش‌بینی کرده باشد، برای ما هر دو 100 هزار دلار خطا هستند. پس قدرمطلق می‌گیریم.

comparison.sort_values("AbsError", ascending=False).head(10) #خانه‌ها را از بیشترین اشتباه تا کمترین اشتباه مرتب می‌کند و ۱۰ مورد اول را نشان می‌دهد.

,Actual,Predicted,Error,AbsError
691,755000,574560.01,180439.99,180439.99
898,611657,457761.18,153895.82,153895.82
1046,556581,422522.57,134058.43,134058.43
581,253293,359358.85,-106065.85,106065.85
218,311500,209452.65,102047.35,102047.35
774,395000,306384.26,88615.74,88615.74
1243,465000,377279.96,87720.04,87720.04
1024,287000,361599.85,-74599.85,74599.85
231,403000,329161.46,73838.54,73838.54
607,225000,156834.67,68165.33,68165.33


In [6]:
#بررسی توزیع قیمت‌ها
#تقریباً همه‌ی خانه‌هایی که بیشترین خطا را دارند خانه‌های بسیار گران هستند. 
#چرا این اتفاق افتاده است؟
#دو دلیل یکی اینکه تعداد کم خانه‌های گران باعث می‌شود مدل الگوی آن‌ها را کمتر یاد بگیرد.
#و دیگری اینکه خانه‌های گران ویژگی‌های خاص‌تری دارند و پیش‌بینی قیمت آن‌ها برای مدل دشوارتر است.

#بهتره یکم بیشتر درباره دیتاهام اطلاعات بدست بیارم
y.describe()

#برای هر ستون عددی یک خلاصه آماری می‌سازد.

#count = چند مقدار غیرخالی دارد
#mean = میانگین
#std = انحراف معیار
#50% = میانه (Median)
#25% = به این می‌گویند چارک اول (Q1)
#ینی 25 درصد داده ها از این مقدار کوچکتر یا مساوی اند
#با کمک چارک‌ها (25%، 50%، 75%) 
#و مقایسهٔ آن‌ها با min و max
#می‌توان فهمید داده‌ها معمولاً در چه بازه‌ای متمرکز شده‌اند
#و آیا مقدارهای خیلی دور از بقیه
#(outlier)وجود دارد یا نه

count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64

In [7]:
#25 درصد خانه‌ها قیمتشان کمتر یا مساوی 129,975 دلار است.
# سه چهارم خانه‌ها زیر 214 هزار دلار هستند.
#ولی یک خانه تا 755 هزار دلار قیمت داره
#این نشان می‌دهد که تعداد کمی خانه خیلی گران وجود دارند.
#این‌ها را در آمار Outlier یا دم توزیع (Tail) می‌گویند.

In [8]:
#نتیجه تا اینجا: 
#روی ۲۰٪ داده تست پیش‌بینی کردیم.دیدیم بعضی خانه‌ها مخصوصاً خانه‌های گران، خطای زیادی دارند.

In [9]:
#یه راه گرفتن لگاریتم از قیمت ها قبل از آموزش روی مدل است تا اختلاف بین خانه‌های ارزان و خیلی گران کمتر شود و مدل راحت‌تر الگوها را یاد بگیرد.
#برنده های مسابقه های کگل قبل از آموزش مدل قیمت خانه هارو لوگاریتمش رو حساب میکنن
#چون معیار مسابقه RMSE است
# و RMSE روی خطاهای بزرگ خیلی حساس است
#چون اگه یه قیمتی خیلی از بقیه دور باشه و اشتباه پیش بینی بشه جریمه بزرگی میکنه
#تبدیل لگاریتمی معمولاً کمک می‌کند مدل روی این خانه‌های خیلی گران هم بهتر یاد بگیرد.

#قبل log:
#100k ----------755k

#بعد از log:
#11.5 ----------13.5

import numpy as np
y = np.log1p(y)
#حالا مدل این اعداد جدید را یاد می‌گیرد، نه قیمت خام را.
#مدل اصلاً خبر ندارد که این لگاریتم است.فقط فکر می‌کند این‌ها همان مقادیر هدف (y) هستند.
###########################################################################
#تقسیم دیتا به دو بخش آموزش و تست
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


#انتخاب مدل
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

#آموزش 
model.fit(X_train, y_train)
###################################################################

#دوباره پیش‌بینی بگیر:
y_pred = model.predict(X_test)

#الان هم y_test و هم y_pred لگاریتم قیمت هستند، نه قیمت واقعی.
#اگر بخواهی قیمت واقعی را ببینی، باید لگاریتم را برگردانی:


y_test_real = np.expm1(y_test)#اگر بخواهی قیمت واقعی را ببینی، باید لگاریتم را برگردانی:
y_pred_real = np.expm1(y_pred)

comparison = pd.DataFrame({#حالا جدول مقایسه را با مقادیر واقعی بساز:
    "Actual": y_test_real,
    "Predicted": y_pred_real
})

comparison.head(10)

,Actual,Predicted
892,154500.0,141117.494384
1105,325000.0,326356.392683
413,115000.0,114811.809254
522,159000.0,155074.329733
1036,315500.0,317575.772784
614,75500.0,87792.355084
218,311500.0,210970.400272
1160,146000.0,150272.760818
649,84500.0,85591.396011
887,135500.0,127073.566077


In [10]:
#دوباره خطاها را حساب کنی:
comparison["Error"] = comparison["Actual"] - comparison["Predicted"]
comparison["AbsError"] = comparison["Error"].abs()

comparison.sort_values("AbsError", ascending=False).head(10)

,Actual,Predicted,Error,AbsError
691,755000.0,543256.172074,211743.827926,211743.827926
898,611657.0,450847.346353,160809.653647,160809.653647
1046,556581.0,434660.817806,121920.182194,121920.182194
774,395000.0,292116.820267,102883.179733,102883.179733
581,253293.0,355622.350810,-102329.350810,102329.350810
218,311500.0,210970.400272,100529.599728,100529.599728
1243,465000.0,384946.073162,80053.926838,80053.926838
451,280000.0,200526.618264,79473.381736,79473.381736
607,225000.0,159062.218374,65937.781626,65937.781626
58,438780.0,373588.657484,65191.342516,65191.342516


In [11]:
# با بررسی خانه‌هایی که بیشترین خطا را دارند، می‌توانیم اثر لگاریتم را ارزیابی کنیم.
# مثلاً قبل از لگاریتم برای خانه‌ای با قیمت 755000 دلار، مدل مقدار 570196 دلار
# را پیش‌بینی کرده بود و حدود 184 هزار دلار خطا داشت. بعد از لگاریتم، همان خانه
# 534968 دلار پیش‌بینی شد و خطا به حدود 220 هزار دلار رسید؛ یعنی برای این نمونه
# عملکرد مدل بدتر شد. اما این به معنی بدتر شدن کل مدل نیست، زیرا ممکن است برای
# بعضی خانه‌های دیگر خطا کمتر شده باشد.
#بنابراین از روی چند نمونه یا 10 خانه نمی‌توان درباره مفید بودن لگاریتم نتیجه‌گیری کرد. 
#  برای قضاوت نهایی باید عملکرد کل مدل را با معیارهایی مانند
# Score و Cross Validation قبل و بعد از لگاریتم مقایسه کنیم.

In [12]:
model.score(X_test, y_test)

0.8861896793703549

In [13]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,X,y,cv=5) #دقت کن که y همان y لگاریتم‌گرفته است.
scores.mean()

np.float64(0.8698162116628433)

In [14]:
#از اینجا به بعد، همین نسخه‌ای که روی y لگاریتم گرفته‌ای را نگه دار.
#چون معیار قابل اعتمادتر (Cross Validation) بهتر شده است.
#در پروژه‌های قیمت‌گذاری خانه، آموزش مدل روی لگاریتم SalePrice یکی از روش‌های استاندارد و رایج است.

In [16]:
#مدل جدید حالا که یادگیریش با لگاریتم ایگرگ انجام شده میخوام ببینم حالا چه ویژگی هایی رو مهم میدونه
importance = pd.Series( 
    model.feature_importances_,  
    index=X.columns 
)
pd.set_option("display.float_format", "{:.6f}".format)#برای اینکه عددهارو با نماد علمی نشون نده و اعشاری معمولی بهم بده
#pd.set_option("display.max_rows", None)#برای اینه که اگه طولانی بود وسطش رو حذف نکنه پاندا
#importance.sort_values(ascending=False)#واسه اینه که همه رو نشون بده
importance.sort_values(ascending=False).head(30)#فقط 30 تای اول رو نشون بده

OverallQual                0.543444
GrLivArea                  0.114076
TotalBsmtSF                0.042573
GarageCars                 0.040925
GarageArea                 0.022275
BsmtFinSF1                 0.021560
1stFlrSF                   0.019298
LotArea                    0.014895
YearBuilt                  0.013508
OverallCond                0.009963
CentralAir_N               0.008729
YearRemodAdd               0.008356
CentralAir_Y               0.007844
GarageYrBlt                0.007581
2ndFlrSF                   0.007424
OpenPorchSF                0.006117
BsmtQual_Ex                0.005834
BsmtUnfSF                  0.005724
LotFrontage                0.005712
GarageType_Detchd          0.004750
MoSold                     0.003629
WoodDeckSF                 0.003623
FireplaceQu_No Fireplace   0.002874
Fireplaces                 0.002570
BsmtQual_Gd                0.002551
GarageType_Attchd          0.002384
GarageFinish_Unf           0.002366
MSZoning_RM                0